# H5N1 corpus work on Google Colab

Connects to the Cloud SQL Postgres warehouse from a Colab VM and runs queries against the
X corpus tables (`x_post_unique`, `x_post_sentiment`, `x_raw`).

**Why this differs from the local notebook:** Colab is a VM in Google's cloud. There is no
`localhost`, so `cloud-sql-proxy` has nothing to bind to. The Cloud SQL Python Connector
opens the same IAM-authenticated tunnel from inside the Python process instead. The
instance's public IP still has no authorized networks and must stay that way.

Full prose: `docs/DB_ACCESS.md`.

## 1. Install and authenticate

`authenticate_user()` authenticates **the Google account signed into Colab**. It must be
your `@g.harvard.edu` — the one holding `cloudsql.client`. A personal Gmail fails with a
permission error, not an auth prompt, which is a confusing way to lose ten minutes.

Never upload a service-account key JSON. The Connector needs no key.

In [ ]:
!pip install -q "cloud-sql-python-connector[pg8000]" sqlalchemy pandas

from google.colab import auth
auth.authenticate_user()
print("authenticated")

## 2. Connect

`getpass` prompts at runtime so the password never lands in the saved `.ipynb`. A password
typed into a cell gets committed with the notebook.

In [ ]:
import getpass
import pandas as pd
import sqlalchemy
from google.cloud.sql.connector import Connector

INSTANCE = "harvard-capstone-499102:us-central1:h5n1-pg"
DB_PASSWORD = getpass.getpass("h5n1_app password: ")

connector = Connector()

def _getconn():
    return connector.connect(
        INSTANCE, "pg8000",
        user="h5n1_app",
        password=DB_PASSWORD,
        db="h5n1",
        # IAM DB auth alternative, once enabled on the instance:
        #   user="<you>@g.harvard.edu", enable_iam_auth=True, no password
    )

engine = sqlalchemy.create_engine(
    "postgresql+pg8000://",
    creator=_getconn,
    pool_pre_ping=True,   # Colab idles; this survives a dropped connection
)

print(pd.read_sql("SELECT version()", engine).iloc[0, 0])

## 3. Session defaults

Set these on **every** connection, not once. SQLAlchemy pools connections, so a `SET` on one
connection does not apply to the next.

`TIME ZONE` is not cosmetic: `to_timestamp` parses in the session timezone and every JSON
timestamp in `x_raw` carries `+0000`. Get it wrong and the whole JSON side shifts.

In [ ]:
from sqlalchemy import event, text

@event.listens_for(engine, "connect")
def _set_session_defaults(dbapi_conn, _):
    cur = dbapi_conn.cursor()
    cur.execute("SET TIME ZONE 'UTC'")
    cur.execute("SET work_mem = '64MB'")
    cur.close()

print(pd.read_sql("SHOW timezone", engine).iloc[0, 0])

## 4. Reading

Straightforward. Use `x_post_unique` (540,225 rows, indexed), not the `x_post` view — the
view re-extracts JSON on every scan and takes ~80 seconds just to count.

In [ ]:
pd.read_sql("""
    SELECT corpus,
           count(*) AS posts,
           min(created_at)::date AS first_day,
           max(created_at)::date AS last_day
    FROM x_post_unique
    GROUP BY corpus ORDER BY posts DESC
""", engine)

## 5. Writing and DDL

`read_sql` never commits. Anything that changes the database needs `engine.begin()`, which
commits on clean exit and rolls back on exception.

**Gotcha:** `.sql` files in this repo contain psql meta-commands — `\timing`, `\echo`,
`\copy`. Those are *psql features, not SQL*, and fail through SQLAlchemy. The helper below
strips them.

In [ ]:
import pathlib
import re

def run_sql_file(path, engine):
    """Execute a .sql file, dropping psql meta-commands that SQLAlchemy cannot run."""
    sql = pathlib.Path(path).read_text()
    sql = "\n".join(l for l in sql.splitlines() if not l.strip().startswith("\\"))
    with engine.begin() as conn:
        conn.exec_driver_sql(sql)
    print(f"applied {path}")

# Example of a direct write:
# with engine.begin() as conn:
#     conn.execute(text("CREATE TABLE x_post_geo (post_id text PRIMARY KEY, "
#                       "fips char(5), state char(2), geo_tier text)"))

## 6. Pulling the corpus down in chunks

540k rows of text will not fit comfortably in a free-tier Colab VM if pulled at once, and a
single huge query is also the thing most likely to die on a dropped connection. Chunk it.

In [ ]:
CHUNK = 50_000

frames = []
for chunk in pd.read_sql("""
        SELECT post_id, created_at, lang, post_text, user_location, corpus
        FROM x_post_unique
    """, engine, chunksize=CHUNK):
    frames.append(chunk)
    print(f"  {sum(len(f) for f in frames):,} rows", end="\r")

df = pd.concat(frames, ignore_index=True)
print(f"\n{len(df):,} rows, {df.memory_usage(deep=True).sum()/1e9:.2f} GB")

## 7. Writing results back

`to_sql` defaults to one INSERT per row, which on 540k rows over a tunnel is hours. Always
pass `method="multi"` with a `chunksize`.

Keep `chunksize` at or below ~1000: `method="multi"` packs rows into one statement, and
Postgres caps a statement at 65535 bound parameters. With 4 columns, 1000 rows is 4000
parameters — safe. Widen the table and you must lower the chunk.

In [ ]:
def write_table(df, name, engine, chunksize=1000):
    df.to_sql(name, engine, if_exists="replace", index=False,
              method="multi", chunksize=chunksize)
    n = pd.read_sql(f"SELECT count(*) AS n FROM {name}", engine).n[0]
    print(f"{name}: {n:,} rows")

# write_table(scores, "x_post_geo", engine)

## 8. First real task — re-score the new French posts

`x_post_sentiment` holds 532,537 rows but the corpus now has 540,225. The gap is the French
data pulled on 2026-10-17. Small enough to be a good end-to-end test of this setup.

In [ ]:
todo = pd.read_sql("""
    SELECT u.post_id, u.created_at, u.lang, u.post_text
    FROM x_post_unique u
    LEFT JOIN x_post_sentiment s USING (post_id)
    WHERE s.post_id IS NULL
""", engine)

print(f"{len(todo):,} posts need scoring")
todo.lang.value_counts()

## 9. GPU, and not losing your work

For sentiment scoring, switch the runtime: **Runtime → Change runtime type → T4 GPU**, then
confirm below. Without it, transformer inference over ~8k posts is slow and over the full
540k is impractical.

**Colab VMs are disposable.** They disconnect after ~90 minutes idle and are capped around
12 hours; everything installed or written to local disk vanishes. Push results to the
database or to GCS as you go — do not leave hours of scoring in a dataframe.

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

In [ ]:
# Checkpoint pattern: write each batch back before starting the next, so a
# disconnect costs one batch rather than the whole run.
#
# for i in range(0, len(todo), 2000):
#     batch = todo.iloc[i:i+2000]
#     scored = score(batch)                      # your sentiment function
#     scored.to_sql("x_post_sentiment", engine, if_exists="append",
#                   index=False, method="multi", chunksize=1000)
#     print(f"{i+len(batch):,} / {len(todo):,}")